# Quantization

Notebooks 01 and 02 both came back to the same number: during decode we move $2d^2$ bytes of weights to
do $2d^2$ FLOPs, an arithmetic intensity of $\approx 1$ against a ridge point of $\approx 153$. Batching
raises the FLOPs. Speculative decoding raises the FLOPs. **Quantization attacks the other side of the
ratio: it shrinks the bytes.**

Store a weight in 8 bits instead of 16 and every decode step reads half as much from HBM. In the
memory-bound regime that is close to a free 2x — *if* the matmul itself can run in the low-precision
format. That "if" is the whole story of part 2:

> **Quantization does not always increase throughput.** Some formats have dedicated tensor-core support
> (INT8 on Ampere, FP8 on Hopper, FP4 on Blackwell); those run faster. Everything else is
> dequantized back to fp16 before the matmul — you win on bandwidth and pay in dequantization overhead.

And unlike speculative decoding, quantization is **lossy**. So the two questions this notebook asks are
exactly the two you will have to answer for any paper that uses it:

1. **Where does accuracy break?** We implement round-to-nearest (RTN) quantization from scratch and sweep
   the bit width until the model falls apart.
2. **What do you actually get?** We run vLLM's natively supported formats and measure memory and
   throughput — at batch 1, where we are memory-bound, and at a large batch, where we are not.


## 0. Setup


In [ ]:
import math
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This notebook needs a GPU."

# point somewhere with a few GB free BEFORE the download cell runs
# os.environ["HF_HOME"] = "/path/with/space/hf_home"

# FlashInfer's sampler JIT-compiles CUDA kernels on first use and needs nvcc.
# Skip it unless you have a CUDA toolkit (not just the runtime torch ships).
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_AWQ = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"  # same model, 4-bit AWQ checkpoint

# What your GPU can actually run: FP8 tensor cores need Hopper (SM 8.9+),
# FP4 needs Blackwell (SM 10.0+). Below that, the format still saves memory
# but the matmul is emulated.
CAP = torch.cuda.get_device_capability()
HAS_FP8 = CAP >= (8, 9)
print(f"{torch.cuda.get_device_name(0)}  (SM {CAP[0]}.{CAP[1]})   fp8 tensor cores: {HAS_FP8}")


In [ ]:
# === Plot helpers: shared palette and axis style (collapse me) =========

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID = "#23373b", "#5c6b70", "#dfe3e4"
SERIES = (BLUE, ORANGE, AQUA)

# Marker style shared by every line plot: filled dot with a white halo, so
# overlapping points stay readable.
DOT = dict(marker="o", ms=8, markeredgecolor="white", markeredgewidth=2)


def style_axes(ax, *, title=None, xlabel=None, ylabel=None, legend=None):
    """Apply the tutorial's axis style: light grid, no box, muted ticks."""
    if title:
        ax.set_title(title, color=INK, fontsize=12, loc="left", pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, color=INK)
    if ylabel:
        ax.set_ylabel(ylabel, color=INK)
    ax.grid(True, which="major", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED)
    if legend:
        ax.legend(frameon=False, loc=legend, labelcolor=INK)
    return ax


In [ ]:
# !hf download {MODEL}
# !hf download {MODEL_AWQ}
# !pip install vllm torch transformers matplotlib numpy huggingface_hub


## 1. Round-to-nearest quantization, from scratch

RTN is the simplest thing that works: pick a scale, divide, round, clamp. For a group of weights $w$ we
store $b$-bit integers plus one fp16 scale (and, asymmetrically, one zero point):

$$
s = \frac{\max(w) - \min(w)}{2^b - 1}, \qquad
z = \left\lfloor -\frac{\min(w)}{s} \right\rceil, \qquad
q = \mathrm{clamp}\!\left(\left\lfloor \frac{w}{s} \right\rceil + z,\; 0,\; 2^b - 1\right), \qquad
\hat{w} = (q - z)\, s
$$

The only real design choice is **what counts as a group** — i.e. how many weights share one scale:

| granularity | scales per matrix | overhead | robustness to outliers |
|---|---|---|---|
| per-tensor | 1 | none | worst — one big weight sets $s$ for everything |
| per-channel (per output row) | `out_features` | ~0 | better |
| group-wise (e.g. 128 along the input dim) | `out*in/128` | $32/128 = 0.25$ bit/weight | best |

This is *simulated* (or "fake") quantization: we quantize and immediately dequantize back to bf16, so the
matmuls still run in bf16. That is deliberate — part 1 measures **accuracy only**. Speed is part 2's job,
and it needs kernels we are not going to write here.


In [ ]:
def rtn_quantize(w, bits, group_size=128, symmetric=False):
    """Fake-quantize a weight matrix (out, in) with round-to-nearest.

    group_size = -1 -> one scale for the whole tensor
    group_size =  0 -> one scale per output channel (row)
    group_size =  g -> one scale per g consecutive weights along the input dim
    """
    out_f, in_f = w.shape
    if group_size == -1:
        wg = w.reshape(1, -1)
    elif group_size == 0:
        wg = w.reshape(out_f, in_f)
    else:
        assert in_f % group_size == 0, f"{in_f} not divisible by {group_size}"
        wg = w.reshape(-1, group_size)
    wg = wg.float()

    if symmetric:
        qmax = 2 ** (bits - 1) - 1
        scale = wg.abs().amax(1, keepdim=True).clamp(min=1e-8) / qmax
        q = (wg / scale).round().clamp(-qmax - 1, qmax)
        deq = q * scale
    else:
        qmax = 2**bits - 1
        mn, mx = wg.amin(1, keepdim=True), wg.amax(1, keepdim=True)
        scale = ((mx - mn) / qmax).clamp(min=1e-8)
        zero = (-mn / scale).round()
        q = (wg / scale + zero).round().clamp(0, qmax)
        deq = (q - zero) * scale

    return deq.reshape(out_f, in_f).to(w.dtype)


def effective_bits(bits, group_size, in_features, symmetric=False):
    """Bits per weight including the fp16 scale (and zero point) we must store."""
    if group_size == -1:
        return bits
    g = in_features if group_size == 0 else group_size
    return bits + (16 if symmetric else 32) / g


Now load the model and pick which layers to quantize. Standard practice: every linear inside the
transformer blocks, but **not** the embedding and **not** the `lm_head` — those are disproportionately
sensitive, and they are also where a 4-bit error turns directly into a wrong token.


In [ ]:
dev = "cuda"
tok = AutoTokenizer.from_pretrained(MODEL)
model = (
    AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to(dev).eval()
)

TARGETS = [
    (n, m)
    for n, m in model.named_modules()
    if isinstance(m, nn.Linear) and ".layers." in n  # skips lm_head / embeddings
]
ORIG = {n: m.weight.detach().clone() for n, m in TARGETS}

n_quant = sum(m.weight.numel() for _, m in TARGETS)
n_total = sum(p.numel() for p in model.parameters())
print(f"{len(TARGETS)} linear layers, {n_quant / 1e9:.2f}B of {n_total / 1e9:.2f}B params quantizable "
      f"({n_quant / n_total:.1%})")
print("example:", TARGETS[0][0], tuple(TARGETS[0][1].weight.shape))


@torch.inference_mode()
def apply_rtn(bits, group_size=128, symmetric=False):
    """Quantize every target layer in place; returns the mean relative weight error."""
    errs = []
    for name, mod in TARGETS:
        w = ORIG[name]
        qw = rtn_quantize(w, bits, group_size, symmetric)
        mod.weight.copy_(qw)
        errs.append((qw.float() - w.float()).norm() / w.float().norm())
    return torch.stack(errs).mean().item()


@torch.inference_mode()
def restore():
    for name, mod in TARGETS:
        mod.weight.copy_(ORIG[name])


### A very simple evaluation

Two cheap probes, no dataset downloads:

- **Perplexity** on a held-out paragraph — a smooth, sensitive signal that starts degrading long before
  anything looks visibly wrong.
- **Five factual questions**, greedy-decoded, scored by substring match — a crude proxy for "is the model
  still usable?", which is what actually falls off a cliff.

They fail at different points, and that gap is the interesting part. (For a paper you would use
WikiText-2 perplexity and a real harness — `lm-evaluation-harness` on MMLU/GSM8K. The *shape* of the
curve you get here is the same.)


In [ ]:
TEXT = """
Machine learning inference is dominated by memory movement rather than arithmetic. During autoregressive
decoding, a transformer reads its entire weight matrix from high-bandwidth memory in order to produce a
single token, which means the arithmetic intensity of the operation is close to one floating point
operation per byte. Modern accelerators are designed for intensities two orders of magnitude higher, so
the tensor cores idle while the memory subsystem works at full speed. Techniques such as batching,
speculative decoding, and quantization all address this imbalance, but they do so in different ways and
under different assumptions about the deployment scenario. Batching amortizes the weight read across many
sequences and therefore requires concurrent requests. Speculative decoding verifies several proposed
tokens in a single forward pass and therefore requires a smaller model that agrees with the larger one.
Quantization reduces the number of bytes that must be read in the first place, and therefore requires
that the loss in numerical precision does not damage the quality of the model's predictions.
""".strip()

QA = [
    ("What is the capital of France? Answer in one word.", "paris"),
    ("What is 17 plus 25? Answer with the number only.", "42"),
    ("Which planet is known as the Red Planet? One word.", "mars"),
    ("Who wrote the play Romeo and Juliet? Surname only.", "shakespeare"),
    ("What does GPU stand for? Answer in one short phrase.", "graphics processing unit"),
]

eval_ids = tok(TEXT, return_tensors="pt").input_ids.to(dev)
qa_ids = [
    tok.apply_chat_template(
        [{"role": "user", "content": q}], add_generation_prompt=True, return_tensors="pt"
    ).to(dev)
    for q, _ in QA
]
print(f"perplexity window: {eval_ids.shape[1]} tokens")


@torch.inference_mode()
def perplexity():
    return math.exp(model(eval_ids, labels=eval_ids).loss.item())


@torch.inference_mode()
def qa_score(verbose=False):
    hits = 0
    for ids, (q, want) in zip(qa_ids, QA):
        out = model.generate(
            ids, max_new_tokens=16, do_sample=False, pad_token_id=tok.eos_token_id
        )
        ans = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
        ok = want in ans.lower()
        hits += ok
        if verbose:
            print(f"  [{'ok ' if ok else 'BAD'}] {q[:38]:38s} -> {ans[:44]!r}")
    return hits / len(QA)


restore()
PPL_BF16, QA_BF16 = perplexity(), qa_score(verbose=True)
print(f"\nbf16 baseline: ppl {PPL_BF16:.3f}   qa {QA_BF16:.0%}")


### The sweep: where does it break?

Three granularities, bit widths from 8 down to 2.


In [ ]:
SCHEMES = {
    "per-tensor": -1,
    "per-channel": 0,
    "group-128": 128,
}
BITS = [8, 6, 5, 4, 3, 2]

results = {}
print(f"{'scheme':12s} {'bits':>5} {'eff.bits':>9} {'w-err':>8} {'ppl':>9} {'ppl x':>7} {'qa':>5}")
print("-" * 62)
for name, gs in SCHEMES.items():
    results[name] = []
    for b in BITS:
        werr = apply_rtn(b, group_size=gs)
        ppl, qa = perplexity(), qa_score()
        eb = effective_bits(b, gs, TARGETS[0][1].in_features)
        results[name].append({"bits": b, "eff_bits": eb, "werr": werr, "ppl": ppl, "qa": qa})
        print(f"{name:12s} {b:5d} {eb:9.2f} {werr:8.4f} {ppl:9.3f} {ppl / PPL_BF16:6.2f}x {qa:5.0%}")
    print()
restore()


In [ ]:
COLORS = {"per-tensor": ORANGE, "per-channel": BLUE, "group-128": AQUA}

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

for name, rows in results.items():
    ax.plot([r["bits"] for r in rows], [r["ppl"] for r in rows], "-", lw=1.8, color=COLORS[name], label=name, **DOT)
ax.axhline(PPL_BF16, color=MUTED, lw=1, ls="--")
ax.text(7.9, PPL_BF16 * 1.06, f"bf16 = {PPL_BF16:.2f}", color=MUTED, fontsize=9, ha="right")
ax.axhline(PPL_BF16 * 1.1, color=GRID, lw=8, alpha=0.6, zorder=0)
ax.set_yscale("log")
ax.invert_xaxis()
style_axes(ax, title="Perplexity degrades smoothly", xlabel="bits per weight",
           ylabel="perplexity (log scale)", legend="upper right")

for name, rows in results.items():
    ax2.plot([r["bits"] for r in rows], [100 * r["qa"] for r in rows], "-", lw=1.8, color=COLORS[name], label=name, **DOT)
ax2.axhline(100 * QA_BF16, color=MUTED, lw=1, ls="--")
ax2.set_ylim(-5, 105)
ax2.invert_xaxis()
style_axes(ax2, title="Usability falls off a cliff", xlabel="bits per weight",
           ylabel="QA accuracy (%)")
fig.tight_layout()
plt.show()


What to read off these curves:

- **8 bit is free.** Perplexity moves in the third decimal, QA is unchanged, and this holds for *any*
  granularity — even one scale for the entire tensor. This is why FP8/INT8 deployment is uncontroversial.
- **4 bit needs groups.** Per-tensor 4-bit is already broken; group-128 4-bit is typically within a few
  percent of bf16. The extra $32/128 = 0.25$ bit/weight buys almost all of the accuracy back — which is
  precisely why every production 4-bit format (AWQ, GPTQ, bitsandbytes NF4) is group-wise.
- **3 bit is the edge, 2 bit is gone.** RTN with no calibration data cannot do better here. This is the
  gap that GPTQ (error compensation via the Hessian) and AWQ (scale the salient channels, chosen using
  activation statistics) exist to close — they beat RTN precisely in the 3–4 bit range and are
  indistinguishable from it at 8.
- **The two metrics disagree, and both are right.** Perplexity starts moving one or two bits *before* the
  QA score does. Report perplexity when you want to detect damage; report a task metric when you want to
  claim the model is still usable. A paper that reports only the second at 4 bit is not lying, but it is
  not telling you where the cliff is either.

**Why granularity matters so much: outliers.** One weight 20x larger than its neighbours forces $s$ up for
everything sharing that scale, and the rest of the group collapses onto a handful of levels.


In [ ]:
w = ORIG[TARGETS[0][0]].float()
absw = w.abs()
print(f"layer {TARGETS[0][0]}  shape {tuple(w.shape)}")
print(f"  std          : {w.std():.4f}")
print(f"  max |w|      : {absw.max():.4f}   ({absw.max() / w.std():.1f} sigma)")
print(f"  99.99th pct  : {absw.flatten().quantile(0.9999):.4f}")
print(f"  fraction of weights above 8 sigma: {(absw > 8 * w.std()).float().mean():.2e}")

# How many distinct levels does a group actually use, with vs. without its outlier?
g = w[0, :128]
for label, vec in [("with outlier", g), ("outlier clipped", g.clamp(-3 * g.std(), 3 * g.std()))]:
    s = (vec.max() - vec.min()) / 15  # 4-bit
    used = torch.unique(((vec - vec.min()) / s).round()).numel()
    print(f"  4-bit levels used by one 128-group, {label:15s}: {used:2d} / 16")


**Exercise.** Re-run the sweep with `symmetric=True` (edit `apply_rtn`'s default). Symmetric quantization
drops the zero point — half the metadata — but wastes a level whenever the group is not centred on zero.
At which bit width does that trade stop being worth it?


## 2. Native quantization in vLLM: memory and speed

Part 1 measured accuracy with fake quantization. Nothing there ran any faster — we dequantized back to
bf16 before every matmul. To actually get a speedup you need a kernel that consumes the low-precision
format, and that is what vLLM ships.

Two things to keep separate:

- **Memory** always improves. 4-bit weights are 4-bit weights on disk and in HBM, whatever the kernel
  does afterwards. Fewer bytes for weights also means *more* bytes left for the KV cache, which is often
  the more valuable win.
- **Speed** improves only in the memory-bound regime, and only if a kernel exists. INT8 needs Ampere,
  FP8 needs Hopper, FP4 needs Blackwell. A 4-bit AWQ model on an A100 has no 4-bit tensor cores: the
  kernel dequantizes into fp16 on the fly, so you win bandwidth and pay compute — great at batch 1,
  frequently *slower* than bf16 at batch 64.

Restart the kernel first — the HF model from part 1 is still holding GPU memory — and re-run section 0.


In [ ]:
# Engine settings shared by every configuration below; the quantization format
# itself is what differs, so it is passed at the call site. All three engines
# stay resident (vLLM cannot release one), so they split the card between them.
ENGINE = dict(gpu_memory_utilization=0.3, max_model_len=2048, enable_prefix_caching=False)

# The configurations we compare. `quantization="fp8"` needs no prepared checkpoint:
# vLLM quantizes the bf16 weights per-channel at load time (this is RTN, from part 1,
# with a format the hardware understands). AWQ needs a checkpoint someone else made.
CONFIGS = {
    "bf16": dict(model=MODEL, dtype="bfloat16"),
    "fp8-w8a8": dict(model=MODEL, dtype="bfloat16", quantization="fp8"),
    "awq-w4a16": dict(model=MODEL_AWQ, dtype="float16"),
}
if not HAS_FP8:
    print(f"SM {CAP[0]}.{CAP[1]} has no fp8 tensor cores -- dropping fp8 (it would be emulated).")
    CONFIGS.pop("fp8-w8a8")
print("comparing:", ", ".join(CONFIGS))


### Memory: weights, and what the freed space is for

Checkpoint size on disk is the honest measure of weight memory — those bytes go to HBM essentially
unchanged. The second table is the part people forget: whatever you save on weights becomes KV cache, and
KV cache is what sets your maximum batch size and context length.

$$
\text{KV bytes per token} = 2 \times n_{\text{layers}} \times n_{\text{kv heads}} \times d_{\text{head}}
\times \text{bytes per element}
$$


In [ ]:
cfg = AutoConfig.from_pretrained(MODEL)
KV_BYTES_PER_TOKEN = (
    2 * cfg.num_hidden_layers * cfg.num_key_value_heads
    * (cfg.hidden_size // cfg.num_attention_heads) * 2  # bf16 KV
)


def ckpt_gib(repo):
    p = Path(snapshot_download(repo, allow_patterns=["*.safetensors", "*.bin"]))
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3


TOTAL_HBM = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {TOTAL_HBM:.0f} GiB total,  KV cache: {KV_BYTES_PER_TOKEN / 1024:.1f} KiB/token (bf16)\n")
print(f"{'config':12s} {'weights':>9} {'vs bf16':>8} {'free for KV':>12} {'tokens of KV':>14}")
print("-" * 60)
sizes = {}
for name, c in CONFIGS.items():
    gib = ckpt_gib(c["model"])
    sizes[name] = gib
    free = TOTAL_HBM * 0.9 - gib  # ~90% is what a serving engine would claim
    print(f"{name:12s} {gib:7.2f} G {sizes['bf16'] / gib:7.2f}x {free:10.1f} G "
          f"{free * 1024**3 / KV_BYTES_PER_TOKEN:13,.0f}")

print("\nNote the asymmetry: 4-bit weights are ~4x smaller, but on a 80 GiB card with a 1.5B model")
print("the KV budget barely moves -- weights were never the bottleneck at this scale. Re-run this")
print("cell mentally with a 70B model (140 GiB bf16) and the picture inverts completely.")


### Speed: batch 1 versus a full batch

This is the measurement that decides whether quantization is worth it for *your* workload. We fix the
number of decoded tokens and sweep the batch size — the same axis as notebook 01, for the same reason.


In [ ]:
BATCHES = [1, 8, 64]
DECODE_TOKENS = 128
PROMPT = "Explain why memory bandwidth, not compute, limits LLM inference speed."


def decode_tps(llm, batch, repeats=3):
    """Decode throughput (tok/s) at a given batch size."""
    sp = SamplingParams(temperature=0.0, max_tokens=DECODE_TOKENS, ignore_eos=True)
    prompts = [PROMPT] * batch
    llm.generate(prompts, sp, use_tqdm=False)  # warm-up
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = llm.generate(prompts, sp, use_tqdm=False)
        best = min(best, time.perf_counter() - t0)
    n = sum(len(o.outputs[0].token_ids) for o in out)
    return n / best


# The five factual probes from part 1, now through the real quantized kernels.
QA = [
    ("What is the capital of France? Answer in one word.", "paris"),
    ("What is 17 plus 25? Answer with the number only.", "42"),
    ("Which planet is known as the Red Planet? One word.", "mars"),
    ("Who wrote the play Romeo and Juliet? Surname only.", "shakespeare"),
    ("What does GPU stand for? Answer in one short phrase.", "graphics processing unit"),
]
sp_qa = SamplingParams(temperature=0.0, max_tokens=16)

# Every measurement for a config is taken in one visit, while its engine is
# the one we just built.
tps, answers = {}, {}
for name, c in CONFIGS.items():
    llm = LLM(**c, **ENGINE)
    tps[name] = [decode_tps(llm, b) for b in BATCHES]
    outs = llm.chat(
        [[{"role": "user", "content": q}] for q, _ in QA], sp_qa, use_tqdm=False
    )
    answers[name] = [o.outputs[0].text.strip() for o in outs]

print(f"\n{'config':12s} " + " ".join(f"{'bs=' + str(b):>16s}" for b in BATCHES))
print("-" * 62)
for name in CONFIGS:
    row = " ".join(
        f"{v:9,.0f} ({v / tps['bf16'][i]:4.2f}x)" for i, v in enumerate(tps[name])
    )
    print(f"{name:12s} {row}")


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
x = np.arange(len(BATCHES))
width = 0.8 / len(tps)
palette = {"bf16": MUTED, "fp8-w8a8": BLUE, "awq-w4a16": ORANGE}

for i, (name, vals) in enumerate(tps.items()):
    rel = [v / tps["bf16"][j] for j, v in enumerate(vals)]
    bars = ax.bar(x + i * width, rel, width * 0.92, color=palette.get(name, AQUA), label=name)
    ax.bar_label(bars, fmt="%.2fx", fontsize=8, color=INK, padding=2)

ax.axhline(1.0, color=INK, lw=1)
ax.set_xticks(x + width * (len(tps) - 1) / 2, [f"batch {b}" for b in BATCHES])
style_axes(ax, title="Quantization pays off where you are memory-bound",
           ylabel="decode throughput, relative to bf16", legend="upper right")
fig.tight_layout()
plt.show()


The expected shape, and why:

- **At batch 1 every format wins**, roughly in proportion to how many bytes it removes: we are reading
  weights and doing almost nothing with them, so bytes *are* the latency.
- **At batch 64 the ranking can invert.** With 64 sequences in flight the weight read is already
  amortized — we have moved right along the roofline — and the extra work now matters. FP8 keeps its
  advantage on Hopper because the matmul itself runs in FP8. AWQ often *loses*, because the 4-bit weights
  are unpacked into fp16 on every forward pass and that dequantization is pure overhead once you are
  compute-bound.
- **Anything shaped like an A100 has no FP8 path.** Then INT8 (`w8a8`, e.g. the
  `RedHatAI/...-quantized.w8a8` checkpoints) is the format with real tensor-core support, and FP8 configs
  fall back to emulation — bandwidth savings only.

So the deployment rule is the mirror image of notebook 02's: **speculative decoding and weight-only
quantization are both low-batch techniques.** The format with hardware support (FP8/INT8) is the one that
keeps helping under load.


### Does it still answer correctly?

The same five questions as part 1, now through the real quantized kernels.


In [ ]:
print(f"{'config':12s} {'qa':>5}   answers")
print("-" * 72)
for name in CONFIGS:
    hits = sum(want in a.lower() for a, (_, want) in zip(answers[name], QA))
    print(f"{name:12s} {hits / len(QA):5.0%}   " + " | ".join(a[:16] for a in answers[name]))


### What to take away

- Quantization shrinks the **bytes** side of arithmetic intensity, which is why it helps exactly where
  batching and speculative decoding do: the memory-bound decode phase at low load.
- **8-bit is the safe default.** RTN is enough, no calibration data is needed, accuracy loss is in the
  noise, and on Hopper/Ada it has real tensor-core support so it keeps winning at high batch too.
- **4-bit is a low-batch, memory-constrained trade.** Use a group-wise format (AWQ/GPTQ), expect a small
  but real accuracy cost, and *measure* throughput at your batch size before assuming it is faster.
- **Memory savings and speed savings are different claims.** A 4x smaller checkpoint that runs 0.9x as
  fast at batch 64 is still the right choice if it is what lets the model fit on the card at all.
- **It is lossy — say so.** Unlike speculative decoding, you cannot leave it on for an evaluation run and
  claim the numbers are unchanged. If a paper's ablations were run in 4-bit, that belongs in the paper.

**Exercises.**
1. Implement per-group *symmetric* RTN and compare against asymmetric at 4 and 3 bits. Where does the
   zero point earn its 16 bits back?
2. Add `kv_cache_dtype="fp8"` to an engine config and re-run the memory table. At 32k context and batch
   64, which matters more — quantizing the weights or the KV cache?
3. Load an INT8 `w8a8` checkpoint (e.g. from the `RedHatAI` collection) and add it to `CONFIGS`. On an
   Ampere card, how does it compare to AWQ at batch 64, and does the roofline explain the gap?
4. Combine notebooks: turn on speculative decoding *and* 4-bit quantization at once. The speedups are not
   multiplicative — explain why using the acceptance-rate formula from 02.
